# Lumen — permissions, Vault audit log and agent traces

Observability reference; `./demo.sh` includes these stages. **Bash** provisioning, `workshop` Kubernetes context, `lumen-observability` namespace. Grafana 13.2.4 + Loki 3.7.8 + Tempo 2.10.8 (monolithic mode) + Alloy 1.21.0-rc.0. This release candidate is pinned for the lab because it includes the fix for lost stdout/stderr lines [grafana/alloy#7080](https://github.com/grafana/alloy/pull/7080). Application instrumented with OpenTelemetry 1.45.1. ClusterIP services; browser and OTLP access through port-forwards on `127.0.0.1`. No Ingress or NodePort is published. [Current review of the interface and services](docs/runs/20261022T140230Z-workspace-review/README.md).

[Walkthrough guide with screenshots](chatbot/OBSERVABILITY.md) · [Executed verification](chatbot/docs/observability-verification.json)

Vault had no auditing enabled when this extension started. Evidence exists only from its activation onward; earlier requests are not reconstructed. The current configuration keeps `lumen-stdout/` with HMAC and five filtered raw devices, detailed below, for selected agent, protection, xAI and SPIFFE paths. The application can capture full JWTs, bodies and responses when its Vault raw recorder is enabled (notebook 8). The revocation lab records metadata only for its Vault calls and never captures its native operator bearer; revocation and contract routes use the protected audit output.


## 1. Enable auditing and correlation

Six root devices: the protected `lumen-stdout/` and five raw outputs (`lumen-agent-raw/`, `lumen-protection-raw/`, `lumen-xai-raw/`, `lumen-xai-access-raw/`, `lumen-spiffe-raw/`) with different filters. `traceparent` links the copies to Tempo. It does not change user or agent policies. Alloy reads the logs of the three Vault pods through the Kubernetes API, with read RBAC restricted to the `vault` namespace; it labels the copies as `audit_mode=raw` or `hmac`.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/enable-audit.sh


## 2. Deploy Grafana, Loki, Tempo and Alloy

PVCs: Loki 5 GiB, Tempo 5 GiB, Grafana 1 GiB. Log/trace retention: 7 days; local lab storage, no HA. Grafana admin: `lumen-admin`; random password stored in Vault `verify/secrets/chatbot/observability` and in the Kubernetes Secret `lumen-observability/grafana-admin`. The chatbot cannot read that secret. The script reuses the existing credential. Grafana has a 1536 MiB limit and `GOMEMLIMIT=768MiB`. When started with `demo.sh`, `launchd` restores the port-forwards if their connection drops.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/deploy-observability.sh
for deployment in grafana loki tempo alloy; do
  kubectl --context workshop -n lumen-observability rollout status deployment/$deployment --timeout=180s
done


## 3. Open local ports and start the instrumented application

For the complete deployment from the root:

```bash
./demo.sh
```

The launcher keeps the port-forwards and the application running as user services. The individual scripts are for maintenance; no extra terminals are needed. Configuration is read from `.env`.

`run.sh` uses `OTEL_EXPORTER_OTLP_TRACES_ENDPOINT=http://127.0.0.1:4318/v1/traces`. Bootstrap tokens remain temporary and limited. The provider is chosen in the banner among the loaded OpenAI/xAI/Claude profiles; each one uses its own Vault identity.

LLM capture is enabled by default with `LUMEN_TRACE_CONTENT=1`: messages, responses, tools, arguments and results, with credentials redacted. Independently, `LUMEN_TRACE_VAULT_RAW=1` stores the Vault exchange, JWTs and credentials in clear text. Per-field limit: 65,536 bytes, with truncation flagged; Tempo uses `distributor.max_attribute_bytes=131072`. To go back to metadata only for new traces: `LUMEN_TRACE_CONTENT=0 LUMEN_TRACE_VAULT_RAW=0 bash chatbot/scripts/run.sh`. The audit log has its own switch (section 8).

In Grafana, open an `xai.chat`, `openai.chat` or `anthropic.chat` span and review `lumen.user.message`, `gen_ai.input.messages`, `gen_ai.output.messages` and `gen_ai.tool.definitions`. In `tool.*`, review `gen_ai.tool.call.arguments` and `gen_ai.tool.call.result`. The chat's **View latest trace** link offers readable panels per call. Earlier traces contain no messages. The LangChain callback content is captured, not the exact HTTP body or the model's internal reasoning.

The local viewer offers an interactive diagram with per-service icons, destination filters, request/return arrows and details for each call. In the lab, **Ver valores sensibles / View sensitive values** opens the OAuth and HTTP Basic values (including `client_secret`), JWT-SVID, MySQL credentials and encryption/decryption bodies. New captures stay in memory for ten minutes; raw Vault/SPIFFE captures follow the trace retention. [Full scope](chatbot/OBSERVABILITY.md#valores-de-cada-llamada).

Each Vault step includes **Cómo decide Vault / How Vault decides**, a drop-down independent of the sensitive values. It queries the correlated audit log to show the identity, the OBO actor, the policies that granted access and the recorded RAR. It distinguishes OAuth, OBO, native token, login and public keys; if events are missing or do not identify the cause of a denial, it says so instead of reconstructing historical policies from the current configuration. [Available data and limits](chatbot/OBSERVABILITY.md#cómo-decide-vault).

![Logic and evidence of a Vault decision](docs/screenshots/current/vault-decision.jpg)

**EN / ES** switches titles, explanations, formats and pagination according to the Lumen preference; original values are never translated. The initial language comes from the browser unless a manual choice has been saved. Grafana keeps its own language. [Bilingual check](docs/manual/10-aplicacion.md#idioma-y-comprobaciones-bilingües) · [Evidence from October 20](chatbot/docs/language-verification.json).

- Chat: <http://localhost:8088>
- Dashboard: <http://localhost:3000/d/lumen-iam>
- Look up the Grafana password in the Vault UI: namespace `verify` → `secrets` → `chatbot/observability`. Do not paste it into a notebook cell.

`./demo.sh start` keeps the Grafana, Loki and Tempo port-forwards running with `launchd`. If you run this notebook without the launcher and the next check cannot connect to port 3000, open them in another terminal with `bash chatbot/scripts/observability-forward.sh`.


In [ ]:
%%bash
set -euo pipefail
curl -fsS http://127.0.0.1:8088/health
curl -fsS http://127.0.0.1:3000/api/health
curl -fsS http://127.0.0.1:3100/ready
curl -fsS http://127.0.0.1:3200/ready


## 4. Visual walkthrough per user

In IBM Verify select **Cloud Directory**. Passwords are in `verify/secrets/chatbot/users/{reader,writer,supervisor}`.

| Action | Reader | Writer | Supervisor |
|---|---|---|---|
| Catalog/inventory/masked inbox | Yes | Yes | Yes |
| Save five-step application | No | Yes, with review and confirmation | No |
| Reveal profile, DNI and IBAN | No, 403 | No, 403 | Yes, Transit/Transform |
| Obtain Vault RW role | 403 | 200 | 403 |
| Administrative secret | 403 | 403 | 403 |
| Public search | Autonomous Tavily agent | Autonomous Tavily agent | Autonomous Tavily agent |

1. Reader: browse the catalog and inventory; «Take the next step» blocks locally without calling Vault.
2. «Check limits in Vault» shows nine real OBO requests: RO/RW/admin, Transit encrypt/decrypt and Transform encode/decode of DNI and IBAN. [Updated matrix and screenshots](docs/PERMISSION_PROBES.md).
3. Writer: ask the agent to prepare Luz Noche, continue to the form, accept fictitious suggestions with Tab, review and save. «Applications» shows the shared masked inbox.
4. Reader and writer cannot reveal. Supervisor retrieves the same record but does not obtain SQL RW.
5. «A wider perspective» uses Client Credentials → Agent Registry → Vault/Tavily.

[Results and screenshots from the real rebuild](docs/E2E_OPENAI_VERIFICATION.md).

![Reader: local block](docs/screenshots/current/reader-write-denied.jpg)
![Writer: application saved](docs/screenshots/current/writer-save-success.jpg)



## 5. From the chat to the audit log and the trace

Copy the **Trace ID** from the Activity panel and paste it into the dashboard. Each Vault request produces a `request` and a `response` with the same `request.id`. The `traceparent` header contains the `trace_id` and `span_id` of `vault.read`, also for 403 responses.

In a log: menu → **Show log details** → **TraceID** field → **Abrir traza**. In Tempo: open the `vault.read` span → **Related logs**. The tree shows `deepagents.run`, `xai.chat`, `openai.chat` or `anthropic.chat` calls, tools, STS, Vault and SQL or Tavily queries. OpenAI and Claude add `spiffe.mint_jwt_svid` and their `openai.token_exchange` or `anthropic.token_exchange` exchange. The OIDC login generates a separate trace, with user and role, once its signature has been validated.

![Current Supervisor audit](docs/screenshots/current/grafana-audit.jpg)
![OpenAI, SPIFFE and OBO catalog in Tempo](docs/screenshots/current/grafana-openai-trace.jpg)

This measures calls made from the chatbot: it does not instrument the internal servers of xAI/OpenAI, Verify, Vault or Tavily, nor does it record the model's private reasoning. The Vault audit log attests to credential issuance/denial; SQL statements are observed through client spans. Independent SQL auditing requires MySQL's own audit facility.


In [ ]:
%%bash
set -euo pipefail
# Paste a recent Trace ID from the interface.
TRACE_ID=''
[ -n "$TRACE_ID" ] || { echo 'Paste a recent Trace ID and rerun the cell.'; exit 0; }
curl -fsS -G http://127.0.0.1:3100/loki/api/v1/query_range \
  --data-urlencode "query={service=\"vault-audit\"} |= \"$TRACE_ID\"" \
  --data-urlencode since=24h --data-urlencode limit=100 \
  | jq '[.data.result[].values[][1]|fromjson|{time,type,request_id:.request.id,path:.request.path,allowed:.auth.policy_results.allowed,policies:.auth.policies,actor:.auth.metadata.actor_entity_name}]|sort_by(.time)'


## 6. Verify correlation and capture mode

The script queries Tempo and Loki and checks the pairs with the same `request.id` and `traceparent`. It prefers the raw copy when available and verifies that the JWTs and responses match the span; for protected records it checks the HMAC. It exports only metadata, booleans and sizes, never credentials.


In [ ]:
%%bash
set -euo pipefail
TRACE_ID=''
[ -n "$TRACE_ID" ] || { echo 'Paste a recent Trace ID and rerun the cell.'; exit 0; }
bash chatbot/scripts/verify-observability.sh "$TRACE_ID"
cd chatbot
/tmp/vault-chatbot-venv/bin/python -m pytest -q tests


## 7. Query the LLM messages and tools

Use the ID of a new query. The end-to-end report compares the captured fields with Tempo and checks that the full form data is not sent to the model. The notebook is distributed without outputs; the next cell shows the content when it runs.

[Historical screenshot: Input text in the Lumen viewer](docs/runs/20261014T162141Z/screenshots/25-model-input.jpg)


In [ ]:
%%bash
set -euo pipefail
TRACE_ID=''
[ -n "$TRACE_ID" ] || { echo 'Paste a recent Trace ID and rerun the cell.'; exit 0; }
curl -fsS "http://127.0.0.1:3200/api/traces/$TRACE_ID" | jq '
  [.batches[]?.scopeSpans[]?.spans[]?
   | select(.name == "xai.chat" or .name == "openai.chat" or .name == "anthropic.chat" or (.name | startswith("tool.")))
   | {name,attributes:[.attributes[]?
       | select(.key | test("^(gen_ai\\.(input.messages|output.messages|tool.definitions|tool.call.arguments|tool.call.result)|lumen\\.(user.message|model.parameters))$"))
       | {field:.key,content:(.value.stringValue|fromjson)}]}]'


## 8. Intercambios completos con Vault y JWT sin HMAC

El modo raw solicitado para el laboratorio conserva estos campos en las llamadas
que utilizan el capturador de Vault de la aplicación. El lab de revocación usa
solo metadatos para sus llamadas a Vault y excluye siempre al operador:

| Campo | Contenido |
|---|---|
| `vault.http.request` | Método, URL, cabeceras de aplicación y cuerpo enviado |
| `vault.auth.jwt` | JWT completo enviado en `X-Vault-Token` |
| `vault.auth.jwt.decoded` | Cabecera y claims: sujeto, grupo, actor, fechas y RAR |
| `vault.http.response` | Estado HTTP, cabeceras y cuerpo, incluidas credenciales o errores |
| `spiffe.jwt_svid` | JWT-SVID completo emitido por Vault |
| `spiffe.jwt_svid.decoded` | Cabecera/claims SPIFFE y validación local de firma |
| `openai.exchange.subject_token` | JWT-SVID enviado al intercambio de OpenAI |
| `anthropic.exchange.assertion` | JWT-SVID enviado al intercambio RFC 7523 de Claude |

El decodificador visual no verifica la firma; la validación ocurre en el cliente OBO y en Vault. En un audit event autenticado, `auth.client_token` conserva el JWT; `request.client_token` puede ser el identificador derivado de Vault. Un rechazo antes de autenticar, por ejemplo por `nbf`, no tiene la misma estructura: no debe confundirse con una denegación de política.

En modo raw se configuran **seis dispositivos de auditoría**:

- `lumen-stdout`: copia HMAC general.
- `lumen-agent-raw`: lecturas RO/RW, Tavily y la prueba de denegación de `verify-admin` en `verify/`.
- `lumen-protection-raw`: seis rutas exactas de Transit/Transform de solicitudes.
- `lumen-xai-raw`: `verify/xai/creds/lumen`.
- `lumen-xai-access-raw`: autorización de la clave xAI en caché, `verify/secrets/data/chatbot/xai-access`.
- `lumen-spiffe-raw`: emisión del JWT-SVID en los roles `lumen-openai` y `lumen-anthropic`.

Los filtros seleccionan rutas y namespace, no identidades. Alloy añade `audit_mode`, sin convertir tokens en etiquetas. Los eventos raw y spans contienen valores en claro por diseño; los informes públicos de verificación solo guardan metadatos, tamaños y booleanos.

`.env` controla `LUMEN_VAULT_AUDIT_MODE=raw` y `LUMEN_TRACE_VAULT_RAW=1`. La captura de mensajes depende de `LUMEN_TRACE_CONTENT=1`. El límite local por campo es 64 KiB, reducible mediante `LUMEN_TRACE_CONTENT_MAX_BYTES`; Tempo admite atributos de 128 KiB. El archivo auxiliar `/tmp/lumen-telemetry/spans.jsonl` no rota. Los tests unitarios usan otro directorio temporal.

Para ocultar valores en nuevas ejecuciones, cambiar `.env` a `LUMEN_VAULT_AUDIT_MODE=hmac` y `LUMEN_TRACE_VAULT_RAW=0` y repetir `./demo.sh start`. Desactivar además `LUMEN_TRACE_CONTENT` si se quieren solo metadatos del modelo. Esto no transforma ni elimina los registros ya almacenados.

Referencias: [auditoría Vault](https://developer.hashicorp.com/vault/docs/audit), [opciones raw](https://developer.hashicorp.com/vault/api-docs/system/audit), [filtros](https://developer.hashicorp.com/vault/docs/enterprise/audit/filtering).

## 9. Solicitudes protegidas: Transit y Transform

El [notebook 8](8_Lumen_Protected_Applications.ipynb) documenta claves, transformaciones, políticas y roles. Guardar: SQL para comprobar referencia, Transit encrypt, dos encode y SQL RW para INSERT. Revelar: SQL RO, decrypt y dos decode. Las llamadas `vault.write` incluyen el cuerpo real: Base64 del perfil, ciphertext o token y transformación.

La comprobación de almacenamiento confirma ciphertext y tokens en MySQL, cuatro tablespaces cifrados y `keyring_okv` activo. El supervisor recuperó exactamente los valores introducidos por el gestor. Ninguna herramienta del agente ofrece descifrado; las consultas del modelo siguen enmascaradas aunque el usuario sea supervisor.

## 10. Credenciales dinámicas de xAI

`chatbot-xai` usa Client Credentials con RAR obligatorio → autorización en `secrets/data/chatbot/xai-access` → `xai/creds/lumen?ttl=300` cuando necesita una clave. Se reutiliza la clave en memoria si Vault permite la lectura de autorización, coincide la generación y quedan más de 210 s. El plugin no renueva leases: la sustitución cambia la clave y el lease. Vault elimina la anterior al caducar; no hay fallback estático. [Bloqueos por CLI y comprobación con JWT ya emitidos](docs/ACCESS_BLOCKS.md).

`model.credentials` registra `cache_hit`, `refresh_reason`, lease anterior/nuevo y vigencia restante. La prueba de 60 s confirma inferencia real y rechazo de la clave caducada por xAI. La prueba histórica de interfaz del 14/10 confirma reutilización y sustitución **antes** de expirar, con ambos eventos correlacionados con Vault. La reconstrucción del 15/10 repitió los 12 checks de ciclo de vida y ocho de RAR/bloqueo; sus informes conservan la fecha de aquella ejecución. Los resultados de la instalación actual se enlazan en la revisión del 22/10.

## 11. Aislamiento de espacios personales

Las operaciones de **Mi espacio de secretos** registran el sujeto OBO, el agente, la entidad propietaria solicitada y la política `lumen-personal-user`. El diagrama distingue lectura, escritura y listado, enlaza el resultado de Vault y conserva la consola de repetición. El contenido personal no se envía al modelo. La auditoría general mantiene HMAC; no se amplían los filtros raw existentes. [Recorrido](docs/manual/16-espacios-personales.md).

El panel **Aislamiento del espacio personal** explica la creación de la entidad,
el vínculo `issuer + external_id → canonical_id`, la ruta KV por UUID y la
resolución de las identidades humana y agente desde el JWT OBO. Compara la
plantilla humana expandida con la ruta solicitada y distingue la emisión del
token en Verify del resultado de la llamada a Vault. Incluye el motivo del
wildcard del Registry y la intersección con política humana y RAR, en ES/EN.

Las nuevas trazas añaden `personal.subject`, `personal.issuer`, nombres del
usuario y destinatario, `personal.actor_client_id`, nombre del techo personal y
snapshots HCL del catálogo. Son metadatos explicativos; no capturan nuevos tokens,
contraseñas ni valores de secretos. La proyección del RAR en este panel conserva
solo tipo, ruta y capacidades de la ruta seleccionada. Los intercambios completos
mantienen sus controles sensibles existentes. La expansión de la plantilla no
consulta ni evalúa la ACL efectiva: el catálogo puede diferir de políticas
modificadas después. Las trazas históricas no se rellenan con datos actuales y
el resultado HTTP real se muestra por separado de la comparación de UUID.


In [ ]:
%%bash
set -euo pipefail
# Deliberately shows JWTs and responses in clear text. Do not save these outputs to the repository.
TRACE_ID=''
[ -n "$TRACE_ID" ] || { echo 'Paste a recent Trace ID and rerun the cell.'; exit 0; }
curl -fsS "http://127.0.0.1:3200/api/traces/$TRACE_ID" | jq '
  [.batches[]?.scopeSpans[]?.spans[]? | select(.name=="vault.read" or .name=="vault.write") |
   {name,attributes:[.attributes[]? | select(.key | test("^vault\\.(http\\.(request|response)|auth\\.jwt(\\.decoded)?)$")) |
      {field:.key,content:(.value.stringValue|fromjson)}]}]'


## Scope and operation

- All auditable requests after the device is enabled are captured, including administrative ones. Some status/health paths do not generate audit entries by design. Blocked local actions and traffic to other services are not Vault requests.
- Grafana is an operator console; its credentials are not given to chatbot users. Loki/Tempo are ClusterIP services without their own authentication, reached through localhost port-forwards. The model/tool fields redact credentials; `LUMEN_TRACE_VAULT_RAW=1` mode deliberately keeps Vault JWTs and responses in clear text.
- Each chat action has its own trace. External searches use predefined public topics. No private MySQL data is sent to Tavily.
- Local `spans.jsonl` in `/tmp/lumen-telemetry`, directory 700/file 600; Tempo keeps traces for 7 days. This auxiliary file is not rotated and should be deleted at the end of the lab.
- Alloy collects stdout; a prolonged outage or container log rotation can lose pending events. This configuration is not immutable production audit storage. For that goal: a redundant durable sink, retention/rotation and external storage.
- If you change the Loki/Tempo/Alloy/datasource ConfigMaps, restart only the affected deployment. Grafana dashboards reload automatically. Repeating the deployment does not restart services.

Sources: [Vault auditing](https://developer.hashicorp.com/vault/docs/audit), [Audit headers](https://developer.hashicorp.com/vault/api-docs/system/config-auditing), [OpenTelemetry Python](https://opentelemetry.io/docs/languages/python/instrumentation/), [Tempo/Loki correlation](https://grafana.com/docs/grafana/latest/datasources/tempo/configure-tempo-data-source/configure-trace-to-logs/), [Alloy Kubernetes](https://grafana.com/docs/alloy/latest/reference/components/loki/loki.source.kubernetes/).
